# Chapter 8: The Awakening
## Companion Notebook — "Eyes of the Machine"

Watch a machine learn for the first time. No math. Just patterns.

---
### 8A: Setup — A Simple Fruit Dataset
We create synthetic "apples" (high red, low blue) and "blueberries" (high blue, low red).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier

# Create synthetic fruit data.
np.random.seed(42)  # Fix randomness so results are reproducible.

apples_red = np.random.uniform(7, 10, 20)
apples_blue = np.random.uniform(0, 3, 20)

blueberries_red = np.random.uniform(0, 3, 20)
blueberries_blue = np.random.uniform(7, 10, 20)

X = np.zeros((40, 2))
X[:20, 0] = apples_red
X[:20, 1] = apples_blue
X[20:, 0] = blueberries_red
X[20:, 1] = blueberries_blue

y = np.zeros(40)
y[20:] = 1

plt.figure(figsize=(6, 6))
plt.scatter(X[:20, 0], X[:20, 1], color='red', label='Apple', s=100)
plt.scatter(X[20:, 0], X[20:, 1], color='blue', label='Blueberry', s=100)
plt.xlabel("Redness")
plt.ylabel("Blueness")
plt.title("Fruit Dataset: Apples vs Blueberries")
plt.legend()
plt.grid(True)
plt.show()

---
### 8B: Training the First Model
KNN: "A fruit is what its 3 nearest neighbors are."

In [ ]:
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X, y)

print("Training complete! The model has learned the fruit pattern.")

---
### 8C: Making a Prediction
Test on a new fruit: redness=8, blueness=2.

In [ ]:
new_fruit = np.array([[8, 2]])
prediction = model.predict(new_fruit)
confidence = model.predict_proba(new_fruit)

if prediction[0] == 0:
    print("Prediction: This is an APPLE!")
else:
    print("Prediction: This is a BLUEBERRY!")

print(f"Confidence: Apple={confidence[0][0]:.1%}, Blueberry={confidence[0][1]:.1%}")

---
### 8D: Visualizing the Decision Boundary
See the regions the model considers "apple" vs "blueberry."

In [ ]:
xx, yy = np.meshgrid(np.arange(0, 11, 0.1), np.arange(0, 11, 0.1))
grid_points = np.c_[xx.ravel(), yy.ravel()]
grid_pred = model.predict(grid_points)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, grid_pred.reshape(xx.shape), alpha=0.3, cmap='RdBu')
plt.scatter(X[:20, 0], X[:20, 1], color='red', label='Apple', s=80, edgecolors='black')
plt.scatter(X[20:, 0], X[20:, 1], color='blue', label='Blueberry', s=80, edgecolors='black')
plt.scatter(new_fruit[0, 0], new_fruit[0, 1], color='green', s=200, marker='*', label='New Fruit', edgecolors='black')
plt.xlabel("Redness")
plt.ylabel("Blueness")
plt.title("Decision Boundary: How the Model 'Sees' the Fruit World")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("The colored region = what the model predicts for ANY fruit here.")

---
### 8E: Overfitting Demo — k=1 vs k=5 vs k=30
Compare how the number of neighbors changes the decision boundary.

In [ ]:
plt.figure(figsize=(15, 4))

for i, k in enumerate([1, 5, 30]):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X, y)
    grid_pred = knn.predict(grid_points)

    plt.subplot(1, 3, i + 1)
    plt.contourf(xx, yy, grid_pred.reshape(xx.shape), alpha=0.3, cmap='RdBu')
    plt.scatter(X[:20, 0], X[:20, 1], color='red', s=30)
    plt.scatter(X[20:, 0], X[20:, 1], color='blue', s=30)
    plt.title(f"k = {k}")
    plt.xlabel("Redness")
    plt.ylabel("Blueness")
    plt.grid(True, alpha=0.3)

plt.suptitle("Overfitting (k=1)  vs  Good Fit (k=5)  vs  Underfitting (k=30)", fontsize=14)
plt.tight_layout()
plt.show()

print("k=1: Wiggly boundary = MEMORIZED (overfitting)")
print("k=5: Smooth pattern = LEARNED (good fit)")
print("k=30: Too smooth = MISSED details (underfitting)")

---
### 8F: From Features to Prediction — Real Images
Extract features from uploaded images and prepare for ML.

In [ ]:
import cv2 as cv
from google.colab import files

print("Upload 3 cat images and 3 dog images:")
uploaded = files.upload()

print("\nExtracting features from each image...")
for filename in uploaded.keys():
    img_bgr = cv.imread(filename)
    img_rgb = cv.cvtColor(img_bgr, cv.COLOR_BGR2RGB)
    img_gray = cv.cvtColor(img_bgr, cv.COLOR_BGR2GRAY)

    avg_brightness = np.mean(img_gray)
    edges = cv.Canny(img_gray, 50, 150)
    edge_density = np.sum(edges > 0) / edges.size
    avg_red = np.mean(img_rgb[:, :, 0])

    print(f"{filename}: Brightness={avg_brightness:.0f}, Edges={edge_density:.2%}, Red={avg_red:.0f}")

---
## 🧠 Challenge: The "Coffee Oracle" — Tired vs Awake Classifier

Take 5 photos of yourself before coffee (tired) and 5 after coffee (awake).
Extract features and train a KNN classifier to tell them apart.
Test on a new photo.

**This is a real ML project. If it works (or fails spectacularly), you've learned.**

In [ ]:
# --- THE COFFEE ORACLE ---
# Step 1: Upload 10 photos (5 tired, 5 awake) — or use any 2-class images.
# Step 2: Fill in the labels manually.
# Step 3: Train KNN.
# Step 4: Test on a new photo.

print("Upload your training photos (10 total):")
training_photos = files.upload()

# You'll need to label them yourself — this is real data science!
print("\nLabel your photos in the array below.")
print("0 = class A (e.g., tired), 1 = class B (e.g., awake)")

# Build feature array and label array.
X_custom = []

for filename in training_photos.keys():
    img = cv.imread(filename)
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)

    feat1 = np.mean(gray)  # Avg brightness.
    edges = cv.Canny(gray, 50, 150)
    feat2 = np.sum(edges > 0) / edges.size * 100  # Edge density.
    feat3 = np.mean(rgb[:, :, 0])  # Avg red.

    X_custom.append([feat1, feat2, feat3])

X_custom = np.array(X_custom)
print(f"Feature matrix shape: {X_custom.shape}")
print("\nNow EDIT the y_custom array below to match your labels.")
print("Example: y_custom = [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]")